# Оценка поиска на готовом датасете

Используем готовый набор из 60 вопросов и ответов для оценки метрик поиска (Recall@1, Recall@3, Recall@5, MRR).

**Датасет:** `data/qa_pairs_v2_podpunkti.jsonl` с вопросами по документу технологической инструкции.

In [1]:
# Настройки
PROJECT_ROOT = "c:/Users/kozlovaalek/Projects/LLM"
JSONL_PATH   = f"{PROJECT_ROOT}/data/02_normalized_text"  # путь к папке с текстами
COLLECTION   = "v2_podpunkti"                               # коллекция Qdrant
EMBED_MODEL  = "bge-m3:latest"
OLLAMA_URL   = "http://localhost:11434"    # Ollama локально или в Docker
QDRANT_URL   = "http://localhost:6333"     # Qdrant локально или в Docker
TOP_K        = 5
SAMPLE       = 0   # 0 = все чанки, иначе случайная выборка N штук

In [2]:
import os
import json, random, time
import requests
import numpy as np
import pandas as pd
from IPython.display import display

print("Setup complete")

Setup complete


In [3]:
import json, random, time
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# Загрузка файлов
items = []
data_dir = Path(JSONL_PATH)

# Рекурсивно ищем все page_*.json файлы
for json_file in sorted(data_dir.glob("**/page_*.json")):
    try:
        with open(json_file, encoding="utf-8") as f:
            page_data = json.load(f)
            # Используем текст страницы как title, если нет отдельного title
            items.append({
                "id": str(json_file),
                "title": f"Page {page_data.get('page', '?')}",
                "text": page_data.get("text", ""),
            })
    except Exception as e:
        print(f"Ошибка загрузки {json_file}: {e}")

# Фильтруем: нужны чанки с текстом
items = [it for it in items if it.get("text", "").strip()]

if SAMPLE and SAMPLE < len(items):
    random.seed(42)
    items = random.sample(items, SAMPLE)

print(f"Чанков для оценки: {len(items)}")

Чанков для оценки: 0


In [4]:
def embed(text: str) -> list:
    r = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": EMBED_MODEL, "input": text},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["embeddings"][0]

def search(vector: list, collection: str, top_k: int) -> list:
    r = requests.post(
        f"{QDRANT_URL}/collections/{collection}/points/search",
        json={"vector": vector, "limit": top_k, "with_payload": True},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["result"]

# Тест связи
try:
    emb = embed("тест")
    print(f"Ollama OK, dim={len(emb)}")
except Exception as e:
    print(f"Ошибка Ollama: {e}")

try:
    r = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}", timeout=5)
    cnt = r.json()["result"]["points_count"]
    print(f"Qdrant OK, коллекция '{COLLECTION}': {cnt} точек")
except Exception as e:
    print(f"Ошибка Qdrant: {e}")

Ошибка Ollama: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/embed (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000016168502200>: Failed to establish a new connection: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение'))
Ошибка Qdrant: HTTPConnectionPool(host='localhost', port=6333): Max retries exceeded with url: /collections/v2_podpunkti (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000016168503190>: Failed to establish a new connection: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение'))


In [5]:
def recall_at_k(results, k):
    found = sum(1 for r in results if r["rank"] is not None and r["rank"] <= k)
    return found / len(results) if results else 0

def mrr(results):
    rr = [1 / r["rank"] if r["rank"] is not None else 0 for r in results]
    return np.mean(rr) if rr else 0

def ndcg_at_k(results, k):
    """NDCG@K: Normalized Discounted Cumulative Gain"""
    def dcg_at_k(rank):
        if rank is None:
            return 0
        return 1 / np.log2(rank + 1) if rank <= k else 0
    
    def ideal_dcg(length, k):
        return sum(1 / np.log2(i + 2) for i in range(min(length, k)))
    
    dcg = sum(dcg_at_k(r["rank"]) for r in results) / len(results) if results else 0
    idcg = ideal_dcg(len(results), k)
    return dcg / idcg if idcg > 0 else 0

---
## Загрузка и оценка на готовом датасете

Загружаем 60 готовых вопросов и ответов из файла `qa_pairs_v2_podpunkti.jsonl`

In [6]:
import os

QA_PATH = os.path.join(PROJECT_ROOT, f"data/qa_pairs_{COLLECTION}.jsonl")

# Загружаем готовые QA пары
qa_pairs = []
if os.path.exists(QA_PATH):
    with open(QA_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                qa_pairs.append(rec)
    print(f"Загружено {len(qa_pairs)} QA-пар из {QA_PATH}")
else:
    print(f"Файл не найден: {QA_PATH}")

print(f"Всего вопросов для оценки: {len(qa_pairs)}")

Файл не найден: c:/Users/kozlovaalek/Projects/LLM\data/qa_pairs_v2_podpunkti.jsonl
Всего вопросов для оценки: 0


In [7]:
# Оценка поиска по готовым вопросам из датасета
qa_results = []
qa_errors = 0

for pair in qa_pairs:
    query = pair["question"]
    answer = pair["answer"]
    question_id = pair["id"]

    try:
        vec = embed(query)
        hits = search(vec, COLLECTION, TOP_K)
    except Exception as e:
        qa_errors += 1
        print(f"Ошибка для {question_id}: {e}")
        continue

    rank = None
    # Проверяем, нашелся ли ответ в результатах поиска по ключевым словам из ответа
    answer_keywords = answer.split()[:10]  # первые 10 слов ответа
    
    for k, h in enumerate(hits, start=1):
        payload = h.get("payload", {})
        hit_text = payload.get("text", "")
        # Проверяем совпадение по ключевым словам
        if any(keyword.lower() in hit_text.lower() for keyword in answer_keywords if len(keyword) > 3):
            rank = k
            break

    qa_results.append({
        "question_id": question_id,
        "question_type": pair.get("type", "unknown"),
        "query": query,
        "answer": answer[:100],
        "rank": rank,
        "multi_hop": pair.get("multi_hop_required", False),
    })

print(f"Оценено: {len(qa_results)}, ошибок: {qa_errors}")
print(f"Вопросов с ответом найдено: {sum(1 for r in qa_results if r['rank'] is not None)}/{len(qa_results)}")

Оценено: 0, ошибок: 0
Вопросов с ответом найдено: 0/0


In [8]:
# Метрики поиска по готовому датасету
qa_metrics = {
    "Вопросов оценено": len(qa_results),
    "Recall@1":         round(recall_at_k(qa_results, 1), 4),
    "Recall@3":         round(recall_at_k(qa_results, 3), 4),
    "Recall@5":         round(recall_at_k(qa_results, 5), 4),
    "MRR":              round(mrr(qa_results), 4),
    "NDCG@5":           round(ndcg_at_k(qa_results, 5), 4),
    "Не найдено":       sum(1 for r in qa_results if r["rank"] is None),
}

display(pd.DataFrame([qa_metrics]).T.rename(columns={0: "Значение"}))

# Анализ по типам вопросов
qa_df = pd.DataFrame(qa_results)
print("\n=== Анализ по типам вопросов ===")
for qtype in sorted(qa_df["question_type"].unique()):
    mask = qa_df["question_type"] == qtype
    count = mask.sum()
    found = (qa_df[mask]["rank"].notna()).sum()
    recall_1 = round(recall_at_k(qa_df[mask].to_dict('records'), 1), 3)
    print(f"{qtype:20} {found:2}/{count:2} найдено | Recall@1: {recall_1}")

,Значение
Вопросов оценено,0
Recall@1,0
Recall@3,0
Recall@5,0
MRR,0
NDCG@5,0
Не найдено,0



=== Анализ по типам вопросов ===


KeyError: 'question_type'

In [ ]:
# Сравнение результатов
print("=== Итоговые метрики ===\n")
print("Self-Retrieval (по title):")
for k, v in metrics.items():
    if k != "Чанков оценено":
        print(f"  {k}: {v}")

print("\nПоиск по готовому датасету (60 вопросов):")
for k, v in qa_metrics.items():
    if k != "Вопросов оценено":
        print(f"  {k}: {v}")

# Примеры не найденных вопросов
print("\n=== Примеры вопросов, где ответ не найден (top-5) ===")
not_found = qa_df[qa_df["rank"].isna()].head(5)
for idx, row in not_found.iterrows():
    print(f"\nID: {row['question_id']}")
    print(f"Тип: {row['question_type']}")
    print(f"Вопрос: {row['query'][:100]}...")
    print(f"Ответ: {row['answer'][:100]}...")

=== Итоговые метрики ===

Self-Retrieval (по title):
  Recall@1: 0.0
  Recall@3: 0.0
  Recall@5: 0.0
  MRR: 0.0
  Не найдено: 126

Поиск по готовому датасету (60 вопросов):
  Recall@1: 0.4118
  Recall@3: 0.7176
  Recall@5: 0.8
  MRR: 0.5665
  NDCG@5: 0.2121
  Не найдено: 17

=== Примеры вопросов, где ответ не найден (top-5) ===

ID: qa_008
Тип: reagent
Вопрос: Для чего используется кальцинированная сода и в каком виде она поступает в ГМУ-1?...
Ответ: Кальцинированная сода обладает нейтрализующими свойствами и применяется в виде водного раствора с ко...

ID: qa_009
Тип: comparison
Вопрос: Чем отличается кислота промывная очищенная от кислоты промывной безметальной по применению?...
Ответ: Обе кислоты описаны как жидкость зелёного цвета с концентрацией кислоты от 15% до 40% и поступают в ...

ID: qa_013
Тип: material
Вопрос: Для чего используются чехлы из лавсана на сектор дискового вакуумного фильтра?...
Ответ: Чехлы из лавсана используются как фильтрующая перегородка на секторах дисков